```markdown
# CTE 442: Applied Artificial Intelligence
## Group 8 Practical Project

### Project Overview
In this project, we design, implement, and benchmark a complete Question Answering pipeline evaluated over several paradigms on the Yaba College of Technology (YABATECH) Official Policies Handbook.

We systematically test and compare:
1. **Zero-Shot Prompting Only**: Directly querying the pre-trained foundation model.
2. **Sparse Retrieval (RAG-BM25)**: Querying with a lexical-search context provider.
3. **Dense Retrieval (RAG-Dense)**: Using a vector database with neural embedding similarity (FAISS + SentenceTransformers).
4. **Supervised Fine-Tuning (LoRA)**: Adapting the base LLM on targeted training pairs using Parameter-Efficient Fine-Tuning.

All strategies are evaluated across 5 experimental seeds to calculate robust average Exact Match (EM) and F1-Scores.
```

```markdown
## Step 1: Environmental Setup & Dependency Installation

We install the required libraries to handle retrieval and modeling:
- `transformers` & `accelerate` for loading and running the Qwen model.
- `rank_bm25` for the lexical (sparse) retrieval baseline.
- `faiss-cpu` & `sentence-transformers` for dense indexing and embedding generation.
- `trl` & `peft` for Parameter-Efficient Fine-Tuning (LoRA).
- `torchao` & `bitsandbytes` to handle potential quantization steps.
```

In [14]:
!pip install -q transformers accelerate rank_bm25 pandas sentencepiece faiss-cpu trl
!pip install -U bitsandbytes>=0.46.1
!pip install --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 26.5 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


```markdown
## Step 2: Corpus Ingestion & Baseline System Construction

Here, we prepare our data infrastructure:
1. **Data loading**: We parse the evaluation set (`test.json`) and the official policies database (`YABATECH_Official_Policies_QA_600.csv`).
2. **Lexical Search setup**: We build a sparse inverted index using `BM25Okapi` over our 600 policy chunks.
3. **Model Initialization**: We spin up `Qwen2.5-0.5B-Instruct` in `float16` precision.
4. **Execution Paradigms**: We define `ask_prompt_only(q)` (no retriever) and `ask_rag_bm25(q)` (which fetches the top-3 most lexically-similar chunks as context) to establish our baselines.
```

In [11]:
import json
import pandas as pd
from rank_bm25 import BM25Okapi
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load Data
with open('test.json', 'r') as f:
    test_data = json.load(f)

# Load the full handbook to act as our "database" for chunks
handbook_df = pd.read_csv('YABATECH_Official_Policies_QA_600.csv')

# Safely load corpus by filling any NaN values with an empty string and converting to string type
corpus = handbook_df['evidence'].fillna("").astype(str).tolist()
tokenized_corpus = [doc.split(" ") for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

# 2. Load the Model (Qwen2.5-0.5B-Instruct)
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Define the two methods
def ask_prompt_only(question):
    messages = [
        {"role": "system", "content": "You are an assistant. Answer the question as briefly as possible. If you do not know, reply strictly with 'Not stated.'"},
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=15, temperature=0.1, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()

def ask_rag_bm25(question, top_k=3):
    tokenized_query = question.split(" ")
    top_docs = bm25.get_top_n(tokenized_query, corpus, n=top_k)
    context = "\n".join(top_docs)

    messages = [
        {"role": "system", "content": "You are a strict assistant. Answer ONLY using the provided text. Keep the answer extremely brief. If the text does not contain the answer, reply strictly with 'Not stated.'"},
        {"role": "user", "content": f"Text:\n{context}\n\nQuestion: {question}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=15, temperature=0.1, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()

# 4. Quick Test on the first 3 questions
print("--- QUICK TEST ---")
for i in range(3):
    q = test_data[i]['question']
    expected = test_data[i]['answer']

    print(f"\nQ: {q}")
    print(f"EXPECTED: {expected}")
    print(f"PROMPTING ONLY: {ask_prompt_only(q)}")
    print(f"RAG (BM25): {ask_rag_bm25(q)}")
    print("-" * 50)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

--- QUICK TEST ---

Q: According to the policy on Sexual Harassment, what is the specific regulation regarding: 'Cases...'?
EXPECTED: 45 days
PROMPTING ONLY: The specific regulation under the policy on Sexual Harassment regarding "Cases..."
RAG (BM25): Not stated.
--------------------------------------------------

Q: According to the policy on Committee Refreshment, what is the specific regulation regarding: 'Refreshment...'?
EXPECTED: One Thousand Naira
PROMPTING ONLY: The specific regulation under the policy on Committee Refreshment regarding refreshments is that
RAG (BM25): Not stated.
--------------------------------------------------

Q: According to the policy on Sexual Harassment, what is the specific regulation regarding: 'Cases...'?
EXPECTED: 45 days
PROMPTING ONLY: The specific regulation under the policy on Sexual Harassment regarding "Cases..."
RAG (BM25): No specific regulation is mentioned in the given text regarding cases of sexual harassment.
--------------------------

```markdown
## Step 3: Multi-Seed Robust Benchmarking (Part 1: Zero-Shot vs. BM25-RAG)

We run a 5-seed simulation loop to eliminate stochastic generation variance.
In each run, we measure:
- **Exact Match (EM)**: Absolute normalized string matching.
- **F1 Score**: Precision and recall computed over word tokens.
- **Hallucination Rate**: Flagging instances where the model generates structured answers for unanswerable queries rather than replying with the strict baseline fallback string *'Not stated.'*

Results are exported to `experiment_results_part1.csv`.
```

In [12]:
import string
import transformers
import pandas as pd

# 1. Helper functions for Metrics
def normalize_text(s):
    """Lowercases and removes punctuation for fairer comparison."""
    return s.lower().translate(str.maketrans('', '', string.punctuation))

def calculate_em(prediction, ground_truth):
    """Exact Match (EM) metric."""
    return 1 if normalize_text(prediction) == normalize_text(ground_truth) else 0

def calculate_f1(prediction, ground_truth):
    """Token-level F1 score."""
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(ground_truth).split()

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return 1 if pred_tokens == truth_tokens else 0

    common_tokens = set(pred_tokens) & set(truth_tokens)
    if not common_tokens:
        return 0

    precision = len(common_tokens) / len(pred_tokens)
    recall = len(common_tokens) / len(truth_tokens)
    return 2 * (precision * recall) / (precision + recall)

# 2. The 5-Seed Experiment Loop
seeds = [1, 2, 3, 4, 5]
results = []

# If testing, you can change test_data to test_data[:10] to run a fast check first!
for seed in seeds:
    print(f"\n--- RUNNING SEED {seed} ---")
    transformers.set_seed(seed) # Locks the randomness

    for row in test_data:
        q_id = row['question_id']
        question = row['question']
        ground_truth = row['answer']
        q_type = row['question_type']

        # Method 1: Prompting Only
        pred_prompt = ask_prompt_only(question)
        em_prompt = calculate_em(pred_prompt, ground_truth)
        f1_prompt = calculate_f1(pred_prompt, ground_truth)

        # Hallucination check: if it's unanswerable, did it invent an answer?
        hallucinated_prompt = 1 if (q_type == 'unanswerable' and "not stated" not in pred_prompt.lower()) else 0

        results.append({
            "method": "Prompting Only",
            "seed": seed,
            "question_id": q_id,
            "prediction": pred_prompt,
            "EM": em_prompt,
            "F1": f1_prompt,
            "hallucinated": hallucinated_prompt
        })

        # Method 2: RAG with BM25 (Top-3)
        pred_rag = ask_rag_bm25(question, top_k=3)
        em_rag = calculate_em(pred_rag, ground_truth)
        f1_rag = calculate_f1(pred_rag, ground_truth)
        hallucinated_rag = 1 if (q_type == 'unanswerable' and "not stated" not in pred_rag.lower()) else 0

        results.append({
            "method": "RAG-BM25",
            "seed": seed,
            "question_id": q_id,
            "prediction": pred_rag,
            "EM": em_rag,
            "F1": f1_rag,
            "hallucinated": hallucinated_rag
        })

# 3. Save the results
results_df = pd.DataFrame(results)
results_df.to_csv("experiment_results_part1.csv", index=False)
print("Finished! Saved to experiment_results_part1.csv")


--- RUNNING SEED 1 ---

--- RUNNING SEED 2 ---

--- RUNNING SEED 3 ---

--- RUNNING SEED 4 ---

--- RUNNING SEED 5 ---
Finished! Saved to experiment_results_part1.csv


In [13]:
!pip install -q sentence-transformers

import faiss
import numpy as np
import pandas as pd
import string
import transformers
from sentence_transformers import SentenceTransformer

# 1. Initialize the Dense Embedding Model and FAISS Index
print("Loading embedding model and creating FAISS index...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Encode the entire handbook corpus into vectors
corpus_embeddings = embedder.encode(corpus, convert_to_numpy=True)

# Create a FAISS index for lightning-fast similarity search
dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(corpus_embeddings)
print(f"FAISS index created with {index.ntotal} chunks.")

# 2. Define the Dense RAG Function
def ask_rag_dense(question, top_k=3):
    # Turn the question into a vector
    question_embedding = embedder.encode([question], convert_to_numpy=True)

    # Search the FAISS index for the closest chunks
    distances, indices = index.search(question_embedding, top_k)
    top_docs = [corpus[idx] for idx in indices[0]]
    context = "\n".join(top_docs)

    # Generate the answer using Qwen
    prompt = f"Answer only from the text below. If the answer is not there, reply 'Not stated.'\nText:\n{context}\n\nQuestion: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=50, temperature=0.3, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()

# Metric helpers (re-declared just in case your Colab runtime refreshed)
def normalize_text(s):
    return s.lower().translate(str.maketrans('', '', string.punctuation))

def calculate_em(prediction, ground_truth):
    return 1 if normalize_text(prediction) == normalize_text(ground_truth) else 0

def calculate_f1(prediction, ground_truth):
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(ground_truth).split()
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return 1 if pred_tokens == truth_tokens else 0
    common = set(pred_tokens) & set(truth_tokens)
    if not common: return 0
    prec = len(common) / len(pred_tokens)
    rec = len(common) / len(truth_tokens)
    return 2 * (prec * rec) / (prec + rec)

# 3. Run the 5-Seed Experiment for Dense RAG
seeds = [1, 2, 3, 4, 5]
dense_results = []

for seed in seeds:
    print(f"\n--- RUNNING DENSE RAG SEED {seed} ---")
    transformers.set_seed(seed)

    for row in test_data:
        q_id = row['question_id']
        question = row['question']
        ground_truth = row['answer']
        q_type = row['question_type']

        # Run Dense RAG
        pred_dense = ask_rag_dense(question, top_k=3)
        em_dense = calculate_em(pred_dense, ground_truth)
        f1_dense = calculate_f1(pred_dense, ground_truth)
        hallucinated_dense = 1 if (q_type == 'unanswerable' and "not stated" not in pred_dense.lower()) else 0

        dense_results.append({
            "method": "RAG-Dense",
            "seed": seed,
            "question_id": q_id,
            "prediction": pred_dense,
            "EM": em_dense,
            "F1": f1_dense,
            "hallucinated": hallucinated_dense
        })

# 4. Merge with previous results and save
dense_df = pd.DataFrame(dense_results)
previous_df = pd.read_csv("experiment_results_part1.csv")

# Combine Prompting, BM25, and Dense results into one master file
final_df = pd.concat([previous_df, dense_df], ignore_index=True)
final_df.to_csv("experiment_results_full.csv", index=False)

print("Finished! All RAG metrics saved to 'experiment_results_full.csv'.")

Loading embedding model and creating FAISS index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index created with 600 chunks.

--- RUNNING DENSE RAG SEED 1 ---

--- RUNNING DENSE RAG SEED 2 ---

--- RUNNING DENSE RAG SEED 3 ---

--- RUNNING DENSE RAG SEED 4 ---

--- RUNNING DENSE RAG SEED 5 ---
Finished! All RAG metrics saved to 'experiment_results_full.csv'.


```markdown
## Step 4: Dense Retrieval RAG Pipeline (FAISS + SentenceTransformers)

Rather than relying on literal keyword overlaps (BM25), we construct a Dense Retriever:
- **Embedder**: `all-MiniLM-L6-v2` encodes both corpus passages and user queries into low-dimensional semantic spaces.
- **Index**: We construct a `FAISS` L2 index to dynamically query and retrieve the top 3 semantically-relevant context blocks.
- **Evaluation**: We execute the same 5-seed benchmark loop and append the dense performance metrics to `experiment_results_full.csv`.
```

In [1]:
import json
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

# 1. Load Data & Format
data_files = {"train": "train.json", "validation": "val.json"}
dataset = load_dataset("json", data_files=data_files)

def format_prompt(example):
    context = example['evidence'] if example['evidence'] else "No context available."
    prompt = f"<|im_start|>user\nContext: {context}\nQuestion: {example['question']}<|im_end|>\n<|im_start|>assistant\n{example['answer']}<|im_end|>"
    return {"text": prompt}

dataset = dataset.map(format_prompt)

# 2. Load Base Model directly in float16 (No 4-bit needed for 0.5B!)
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Set up and apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Ensure all trainable LoRA parameters are in float32 so no GradScaler is needed
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
    elif param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

# 4. Training Arguments (fp16=False disables the buggy PyTorch GradScaler)
training_args = SFTConfig(
    output_dir="./yabatech-lora-model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim="adamw_torch",
    fp16=False,                 # <--- Disables the crashing GradScaler
    bf16=False,
    report_to="none",
    dataset_text_field="text",
    max_length=512
)

# 5. Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    args=training_args,
)

# 6. START TRAINING!
print("Starting LoRA Fine-tuning...")
trainer.train()

# 7. Save the final adapter
trainer.model.save_pretrained("./yabatech-lora-final")
tokenizer.save_pretrained("./yabatech-lora-final")
print("Training Complete! LoRA adapter saved to ./yabatech-lora-final")

W1117 22:07:37.723000 26946 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1117 22:07:37.857000 26946 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading model and tokenizer...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Starting LoRA Fine-tuning...


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.828937,1.234959,1.368176,0.774574,22559.000000
2,0.781036,0.538921,0.775401,0.885491,45118.000000
3,0.374602,0.309368,0.549012,0.937193,67677.000000


Training Complete! LoRA adapter saved to ./yabatech-lora-final


```markdown
## Step 5: Parameter-Efficient Fine-Tuning (LoRA SFT)

To embed the domain knowledge directly within the model weights without needing an external database retrieval block at generation runtime, we apply Supervised Fine-Tuning:
- **Configuration**: We load `train.json` and map it into structured prompt-completion pairs matching the ChatML template style.
- **Optimization**: We train a LoRA adapter (`r=16, alpha=32`) targetting the query, key, value, and projection layers of our attention block.
- **SFTConfig**: We disable fp16 mixed-precision scaling to prevent gradient overflows (`fp16=False`), and train using PyTorch's native `adamw_torch` optimizer over 3 epochs.
- **Output**: The finalized adapted weights are saved to `./yabatech-lora-final`.
```

In [2]:
import json
import pandas as pd
import torch
import string
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Load the Test Data
with open('test.json', 'r') as f:
    test_data = json.load(f)

# 2. Load Base Model + Your Trained LoRA Adapter
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, "./yabatech-lora-final")

# 3. Define the Inference Function
def ask_lora(question, context):
    # We must use the exact prompt format the model saw during training!
    ctx = context if context else "No context available."
    prompt = f"<|im_start|>user\nContext: {ctx}\nQuestion: {question}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=15, temperature=0.1, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()

# 4. Metric Helpers
def normalize_text(s):
    return s.lower().translate(str.maketrans('', '', string.punctuation))

def calculate_em(prediction, ground_truth):
    return 1 if normalize_text(prediction) == normalize_text(ground_truth) else 0

def calculate_f1(prediction, ground_truth):
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(ground_truth).split()
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return 1 if pred_tokens == truth_tokens else 0
    common = set(pred_tokens) & set(truth_tokens)
    if not common: return 0
    prec = len(common) / len(pred_tokens)
    rec = len(common) / len(truth_tokens)
    return 2 * (prec * rec) / (prec + rec)

# 5. Run the 5-Seed Evaluation Loop
seeds = [1, 2, 3, 4, 5]
lora_results = []

for seed in seeds:
    print(f"\n--- RUNNING LORA SEED {seed} ---")
    transformers.set_seed(seed)

    for row in test_data:
        q_id = row['question_id']
        question = row['question']
        ground_truth = row['answer']
        evidence = row['evidence']
        q_type = row['question_type']

        # Test the model
        pred_lora = ask_lora(question, evidence)
        em_lora = calculate_em(pred_lora, ground_truth)
        f1_lora = calculate_f1(pred_lora, ground_truth)
        hallucinated_lora = 1 if (q_type == 'unanswerable' and "not stated" not in pred_lora.lower()) else 0

        lora_results.append({
            "method": "LoRA Fine-Tuned",
            "seed": seed,
            "question_id": q_id,
            "prediction": pred_lora,
            "EM": em_lora,
            "F1": f1_lora,
            "hallucinated": hallucinated_lora
        })

# 6. Merge with previous RAG results and save the final master file
lora_df = pd.DataFrame(lora_results)
previous_df = pd.read_csv("experiment_results_full.csv")

final_master_df = pd.concat([previous_df, lora_df], ignore_index=True)
final_master_df.to_csv("experiment_results_MASTER.csv", index=False)

print("\nFinished! All experiments complete. Results saved to 'experiment_results_MASTER.csv'.")

Loading base model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading LoRA adapter...

--- RUNNING LORA SEED 1 ---

--- RUNNING LORA SEED 2 ---

--- RUNNING LORA SEED 3 ---

--- RUNNING LORA SEED 4 ---

--- RUNNING LORA SEED 5 ---

Finished! All experiments complete. Results saved to 'experiment_results_MASTER.csv'.
